In [3]:
#1 – Imports
from pathlib import Path
import pickle
import hashlib
from collections import defaultdict

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
#2 – Project Paths
PROJECT_ROOT = Path("..").resolve()

ARTIFACTS = PROJECT_ROOT / "artifacts"

INPUT_FILE = ARTIFACTS / "documents.pkl"

OUTPUT_FILE = ARTIFACTS / "chunk_documents.pkl"

In [5]:
#3 – Load Documents
with open(INPUT_FILE, "rb") as f:
    documents = pickle.load(f)

print(f"Documents Loaded : {len(documents)}")

Documents Loaded : 773


In [7]:
#4 – Verify
print(type(documents))
print(type(documents[0]))

<class 'list'>
<class 'langchain_core.documents.base.Document'>


In [ ]:
#5 – Configure Text Splitter

In [8]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=200,

    length_function=len,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

In [9]:
#6 – Split Documents
chunk_documents = text_splitter.split_documents(documents)

print(f"Total Chunks : {len(chunk_documents)}")

Total Chunks : 5073


In [11]:
#7 – SHA256 Helper
def sha256(text: str) -> str:
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

In [12]:
#8 – Add Chunk Metadata
chunk_counter = defaultdict(int)

for chunk in chunk_documents:

    document_id = chunk.metadata["document_id"]

    chunk_number = chunk_counter[document_id]

    chunk.metadata["chunk_number"] = chunk_number

    chunk.metadata["chunk_id"] = sha256(
        f"{document_id}|{chunk_number}"
    )

    chunk.metadata["content_hash"] = sha256(
        chunk.page_content
    )

    chunk.metadata["chunk_size"] = len(
        chunk.page_content
    )

    chunk.metadata["word_count"] = len(
        chunk.page_content.split()
    )

    chunk_counter[document_id] += 1

In [ ]:
#9 – Verify Metadata
sample = chunk_documents[0]

sample.metadata

{'document_id': 'bac5cafbe3fad78362bf9f7102a348a004454d67a65cec3cf592deed2ad6d9fb',
 'document_hash': 'f95538b822db9ca84075a7b73f7b4ed08c2e4e087ec3c972c08f9e6565a85924',
 'company': 'claude',
 'product_area': 'amazon_bedrock',
 'title': '"How do I learn more about Amazon and Anthropic’s strategic collaboration?"',
 'url': '"https://support.claude.com/en/articles/10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration"',
 'file_name': '10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration.md',
 'source_path': 'claude\\amazon-bedrock\\10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration.md',
 'extension': '.md',
 'source_type': 'markdown',
 'schema_version': '1.0',
 'indexed_at': '2026-07-07T20:51:17.786235+00:00',
 'last_modified': '2026-07-02T21:53:33.772476+00:00',
 'chunk_number': 0,
 'chunk_id': 'a626375f498a76333041bb81874837aadeb4ef80351c4c9b04582d972e9b7531',
 'content_hash': 'f95538b822db9ca84075a

In [14]:
#10 – Preview Chunk
print("="*80)

print(chunk_documents[0].metadata)

print("="*80)

print(chunk_documents[0].page_content)

{'document_id': 'bac5cafbe3fad78362bf9f7102a348a004454d67a65cec3cf592deed2ad6d9fb', 'document_hash': 'f95538b822db9ca84075a7b73f7b4ed08c2e4e087ec3c972c08f9e6565a85924', 'company': 'claude', 'product_area': 'amazon_bedrock', 'title': '"How do I learn more about Amazon and Anthropic’s strategic collaboration?"', 'url': '"https://support.claude.com/en/articles/10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration"', 'file_name': '10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration.md', 'source_path': 'claude\\amazon-bedrock\\10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration.md', 'extension': '.md', 'source_type': 'markdown', 'schema_version': '1.0', 'indexed_at': '2026-07-07T20:51:17.786235+00:00', 'last_modified': '2026-07-02T21:53:33.772476+00:00', 'chunk_number': 0, 'chunk_id': 'a626375f498a76333041bb81874837aadeb4ef80351c4c9b04582d972e9b7531', 'content_hash': 'f95538b822db9ca84075a7b73f7b4ed08c2e

In [15]:
#11 – Duplicate Chunk IDs
chunk_ids = [
    doc.metadata["chunk_id"]
    for doc in chunk_documents
]

duplicates = len(chunk_ids) - len(set(chunk_ids))

print(f"Duplicate Chunk IDs : {duplicates}")

Duplicate Chunk IDs : 0


In [16]:
#12 – Duplicate Content
content_hashes = [
    doc.metadata["content_hash"]
    for doc in chunk_documents
]

duplicates = len(content_hashes) - len(set(content_hashes))

print(f"Duplicate Content : {duplicates}")

Duplicate Content : 65


In [18]:
#13 – Chunk Size Statistics
chunk_sizes = [
    doc.metadata["chunk_size"]
    for doc in chunk_documents
]

print(f"Minimum : {min(chunk_sizes)}")

print(f"Maximum : {max(chunk_sizes)}")

print(f"Average : {sum(chunk_sizes)/len(chunk_sizes):.2f}")

Minimum : 50
Maximum : 1000
Average : 859.66


In [19]:
#14 – Company Distribution
from collections import Counter

company_counter = Counter(
    doc.metadata["company"]
    for doc in chunk_documents
)

company_counter

Counter({'hackerrank': 3339, 'claude': 1678, 'visa': 56})

In [20]:
#15 – Product Area Distribution
product_counter = Counter(
    doc.metadata["product_area"]
    for doc in chunk_documents
)

product_counter

Counter({'integrations': 862,
         'library': 568,
         'claude': 486,
         'general_help': 463,
         'screen': 446,
         'settings': 335,
         'team_and_enterprise_plans': 288,
         'interviews': 194,
         'claude_code': 145,
         'hackerrank_community': 139,
         'connectors': 137,
         'skillup': 124,
         'claude_api_and_console': 105,
         'uncategorized': 93,
         'claude_mobile_apps': 69,
         'pro_and_max_plans': 65,
         'claude_for_nonprofits': 58,
         'safeguards': 57,
         'index.md': 56,
         'engage': 55,
         'privacy_and_legal': 51,
         'claude_for_government': 48,
         'identity_management_sso_jit_scim': 45,
         'claude_desktop': 42,
         'support': 40,
         'claude_in_chrome': 37,
         'chakra': 24,
         'claude_for_education': 18,
         'support.md': 15,
         'amazon_bedrock': 8})

In [21]:
#16 – Random Sample
import random

sample = random.choice(chunk_documents)

print(sample.metadata)

print()

print(sample.page_content)

{'document_id': 'ab0ebd419e38d474b2c725f7db1d718c54eddf24290bf3c5f3c4494a557d69ea', 'document_hash': 'ed0ba2e900d84f9dd3e371a9badb1104f9a505dd4698d0e4fef5ea3c47014ddb', 'company': 'hackerrank', 'product_area': 'general_help', 'title': '"January 2025 Release Notes"', 'url': '"https://support.hackerrank.com/articles/8074371720-january-2025-release-notes"', 'file_name': '8074371720-january-2025-release-notes.md', 'source_path': 'hackerrank\\general-help\\release-notes\\8074371720-january-2025-release-notes.md', 'extension': '.md', 'source_type': 'markdown', 'schema_version': '1.0', 'indexed_at': '2026-07-07T20:51:18.046578+00:00', 'last_modified': '2026-07-02T21:53:33.932791+00:00', 'chunk_number': 9, 'chunk_id': '6b5aaee48ec9fd054e8e2a04681ccd9619b7b23c65630df2f7f4181030454cce', 'content_hash': '70c9f630e67dce18419a5ec1211d450039a076ad180f60ae56ef56f8d0c7a027', 'chunk_size': 942, 'word_count': 138}

<p><br />
</p>
<p>You are given:</p>
<p>- m: The number of eligible men.</p>
<p>- w: The 

In [22]:
#17 – Save Chunk Documents
with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(chunk_documents, f)

print(f"Saved {len(chunk_documents)} chunk documents")
print(f"Location : {OUTPUT_FILE}")

Saved 5073 chunk documents
Location : C:\Users\cheru\OneDrive\Desktop\GitHub_Projects\SupportSphere-AI\artifacts\chunk_documents.pkl


In [23]:
#18 – Reload Verification
with open(OUTPUT_FILE, "rb") as f:
    loaded_chunk_documents = pickle.load(f)

print(type(loaded_chunk_documents))
print(type(loaded_chunk_documents[0]))
print(len(loaded_chunk_documents))

<class 'list'>
<class 'langchain_core.documents.base.Document'>
5073
